Qwen3 Mixture-of-Experts From Scratch (A Standalone Notebook)

In [ ]:
from importlib.metadata import version
# 中文注释: 导入 version 函数,用于查询已安装第三方库的版本号,便于确认运行环境

# 中文注释: 列出本 notebook 依赖的关键第三方库,逐一打印版本号以便复现实验环境
pkgs = [
    "huggingface_hub",  # to download pretrained weights
    "tokenizers",       # to implement the tokenizer
    "torch",            # to implement the model
]
# 中文注释: 遍历依赖列表,打印每个库的版本
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
import torch
import torch.nn as nn


# 中文注释: FeedForward 实现 SwiGLU 前馈网络(非 MoE 层使用)。fc1、fc2 都把 emb_dim 升维到 hidden_dim,
# 计算 silu(fc1(x)) * fc2(x) 作为门控后的隐藏表示,再由 fc3 把 hidden_dim 降回 emb_dim
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    def forward(self, x):
        # 中文注释: x 形状为 (batch, seq_len, emb_dim);x_fc1、x_fc2 形状均为 (batch, seq_len, hidden_dim)
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = nn.functional.silu(x_fc1) * x_fc2
        return self.fc3(x)


# 中文注释: MoEFeedForward —— 稀疏混合专家(Mixture-of-Experts)前馈层。
# 核心流程: 1) gate 线性层为每个 token 打分(路由); 2) 取 top-k 专家做 softmax 得到聚合权重;
# 3) 仅让被选中的 token 子集经过对应专家的 SwiGLU 计算(稀疏计算,而不是让每个 token 过全部专家);
# 4) 按路由权重把各专家输出加权累加回该 token 的位置
class MoEFeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.num_experts_per_tok = cfg["num_experts_per_tok"]
        self.num_experts = cfg["num_experts"]
        self.emb_dim = cfg["emb_dim"]
        # 中文注释: 门控(路由)线性层,把每个 token 的 emb_dim 表征映射为 num_experts 个专家打分(logits)
        self.gate = nn.Linear(cfg["emb_dim"], cfg["num_experts"], bias=False, dtype=cfg["dtype"])

        # 中文注释: 为每个专家分别创建独立的 fc1(门控投影)/fc2(上投影)/fc3(下投影)三个线性层,
        # 专家之间参数互不共享,是 MoE 用更多参数量换取容量的关键
        self.fc1 = nn.ModuleList([nn.Linear(cfg["emb_dim"], cfg["moe_hidden_dim"], bias=False, dtype=cfg["dtype"])
                                  for _ in range(cfg["num_experts"])])
        self.fc2 = nn.ModuleList([nn.Linear(cfg["emb_dim"], cfg["moe_hidden_dim"], bias=False, dtype=cfg["dtype"])
                                  for _ in range(cfg["num_experts"])])
        self.fc3 = nn.ModuleList([nn.Linear(cfg["moe_hidden_dim"], cfg["emb_dim"], bias=False, dtype=cfg["dtype"])
                                  for _ in range(cfg["num_experts"])])

    def forward(self, x):
        # 中文注释: 计算路由得分(logits),形状 (batch, seq_len, num_experts)
        scores = self.gate(x)  # (b, seq_len, num_experts)
        topk_scores, topk_indices = torch.topk(scores, self.num_experts_per_tok, dim=-1)
        # 中文注释: 为每个 token 选出得分最高的 num_experts_per_tok 个专家及其下标,
        # topk_scores/topk_indices 形状均为 (batch, seq_len, num_experts_per_tok)
        topk_probs = torch.softmax(topk_scores, dim=-1)
        # 中文注释: 只在被选中的 top-k 专家范围内做 softmax 归一化,得到专家聚合权重
        # (注意不是对全部 num_experts 做 softmax,这是 MoE 路由的常见做法)

        # 中文注释: 把 (batch, seq_len, emb_dim) 展平为 (batch*seq_len, emb_dim),
        # 便于按 token 维度把不同 token 分派(dispatch)给不同专家处理
        batch, seq_len, _ = x.shape
        x_flat = x.reshape(batch * seq_len, -1)
        # 中文注释: 输出累加缓冲区,形状 (batch*seq_len, emb_dim),初始化为全 0,
        # 之后用 index_add_ 按 token 下标把各专家的加权输出累加进来
        out_flat = torch.zeros(batch * seq_len, self.emb_dim, device=x.device, dtype=x.dtype)

        # 中文注释: 把 top-k 索引与权重也展平为 (batch*seq_len, num_experts_per_tok),与 x_flat 的 token 维度对齐
        topk_indices_flat = topk_indices.reshape(-1, self.num_experts_per_tok)
        topk_probs_flat = topk_probs.reshape(-1, self.num_experts_per_tok)

        # 中文注释: 统计当前 batch 中实际被选中过的专家 id 集合,避免遍历全部 num_experts,
        # 在专家数量很多、每个 token 只激活少数专家时能显著减少计算
        unique_experts = torch.unique(topk_indices_flat)

        # 中文注释: 对每个被激活的专家,只挑出把它选为 top-k 之一的 token 子集来计算,
        # 从而实现稀疏计算(每个 token 只经过 num_experts_per_tok 个专家,而不是全部专家)
        for expert_id_tensor in unique_experts:
            expert_id = int(expert_id_tensor.item())
            # 中文注释: mask 形状 (batch*seq_len, num_experts_per_tok),
            # 标记所有 token 的每个 top-k 槽位是否恰好等于当前专家 id
            mask = topk_indices_flat == expert_id
            if not mask.any():
                continue

            # 中文注释: 按 token 维度归约,得到哪些 token 选中了当前专家(不关心具体是第几个槽位),
            # 形状 (batch*seq_len,)
            token_mask = mask.any(dim=-1)
            selected_idx = token_mask.nonzero(as_tuple=False).squeeze(-1)
            # 中文注释: selected_idx 是选中了当前专家的 token 在展平序列中的下标,数量为 num_selected
            if selected_idx.numel() == 0:
                continue

            # 中文注释: 只取出这些被选中 token 的输入向量,形状 (num_selected, emb_dim),
            # 仅对路由到当前专家的 token 计算,节省算力
            expert_input = x_flat.index_select(0, selected_idx)
            # 中文注释: 当前专家的 SwiGLU 前馈计算 silu(fc1(x)) * fc2(x),形状 (num_selected, moe_hidden_dim)
            hidden = torch.nn.functional.silu(self.fc1[expert_id](expert_input)) * self.fc2[expert_id](expert_input)
            expert_out = self.fc3[expert_id](hidden)
            # 中文注释: 经 fc3 降维回 emb_dim,得到该专家对这批 token 的输出,形状 (num_selected, emb_dim)

            # 中文注释: 取出被选中 token 对应的 mask 行,形状 (num_selected, num_experts_per_tok)
            mask_selected = mask[selected_idx]
            # 中文注释: 定位当前专家在每个 token 的 top-k 槽位中排第几,用于取出对应的路由权重
            slot_indices = mask_selected.int().argmax(dim=-1, keepdim=True)
            # 中文注释: 按槽位取出 softmax 后的路由权重,形状 (num_selected,)
            selected_probs = torch.gather(topk_probs_flat.index_select(0, selected_idx), dim=-1, index=slot_indices).squeeze(-1)

            # 中文注释: 专家输出乘以对应路由权重后,按 token 下标累加回输出缓冲区
            # (同一个 token 若命中多个专家,会被多次累加,从而实现 top-k 专家的加权聚合)
            out_flat.index_add_(0, selected_idx, expert_out * selected_probs.unsqueeze(-1))

        # 中文注释: 恢复形状为 (batch, seq_len, emb_dim),作为该 MoE 层最终输出
        return out_flat.reshape(batch, seq_len, self.emb_dim)
# 中文注释: RMSNorm(均方根归一化)—— 比 LayerNorm 更轻量,不做去均值(no mean subtraction),
# 只按均方根(RMS)缩放,再乘以可学习的 scale(可选再加 shift),Qwen3 中用于替代 LayerNorm
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False, qwen3_compatible=True):
        super().__init__()
        self.eps = eps
        self.qwen3_compatible = qwen3_compatible
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self, x):
        input_dtype = x.dtype

        # 中文注释: 为了数值稳定性,归一化统计量在 float32 下计算(即使输入是 bfloat16),
        # 计算完成后再转换回原始 dtype
        if self.qwen3_compatible:
            x = x.to(torch.float32)

        # 中文注释: 对最后一维(emb_dim 或 head_dim)求平方均值,形状从 (..., dim) 归约为 (..., 1)
        variance = x.pow(2).mean(dim=-1, keepdim=True)
        norm_x = x * torch.rsqrt(variance + self.eps)
        # 中文注释: 乘以可学习缩放系数 scale(形状与最后一维相同,可广播到 x 的形状)
        norm_x = norm_x * self.scale

        if self.shift is not None:
            norm_x = norm_x + self.shift

        return norm_x.to(input_dtype)
# 中文注释: 预计算 RoPE(旋转位置编码, Rotary Position Embedding)所需的 cos/sin 表,
# 覆盖 0 .. context_length-1 的每一个位置,forward 时直接查表使用
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # Compute the inverse frequencies
    # 中文注释: 逆频率 inv_freq 形状为 (head_dim//2,),频率随维度增大呈指数衰减;
    # theta_base 越大,相邻位置之间的相位变化越平缓,越适合超长上下文
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim))

    # Generate position indices
    # 中文注释: 位置索引 positions 形状为 (context_length,)
    positions = torch.arange(context_length, dtype=dtype)

    # Compute the angles
    # 中文注释: 位置与逆频率做外积,得到每个位置在每个频率上的旋转角度
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (context_length, head_dim // 2)

    # Expand angles to match the head_dim
    # 中文注释: 把角度复制拼接一份,使形状变为 (context_length, head_dim),
    # 以便与完整的 head_dim 向量做逐元素的旋转运算
    angles = torch.cat([angles, angles], dim=1)  # Shape: (context_length, head_dim)

    # Precompute sine and cosine
    # 中文注释: 预先算好 cos、sin 两张表,避免在每次前向传播里重复计算三角函数
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


# 中文注释: 对 query/key 应用 RoPE —— 通过在成对维度上做旋转来编码相对位置信息,
# 相比可学习的绝对位置编码,RoPE 天然具有外推到更长序列的能力
def apply_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    # Split x into first half and second half
    # 中文注释: 把最后一维 head_dim 一分为二,x1 为前半部分,x2 为后半部分,
    # 形状均为 (batch_size, num_heads, seq_len, head_dim//2)
    x1 = x[..., : head_dim // 2]  # First half
    x2 = x[..., head_dim // 2 :]  # Second half

    # Adjust sin and cos shapes
    # 中文注释: 根据当前序列长度截取对应位置的 cos/sin,并各自增加 batch 和 head 两个维度用于广播
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    # 中文注释: 构造“旋转后”的向量 [-x2, x1],再与 cos/sin 组合,
    # 实现二维平面上的旋转: x_rotated = x * cos + rotate_half(x) * sin
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    # It's ok to use lower-precision after applying cos and sin rotation
    return x_rotated.to(dtype=x.dtype)
# 中文注释: 分组查询注意力(GQA, Grouped Query Attention)—— 多个 query 头共享同一组 key/value 头,
# 用于减少 KV 缓存的显存占用;当 num_kv_groups == num_heads 时退化为标准多头注意力(MHA),
# 当 num_kv_groups == 1 时退化为多查询注意力(MQA)
class GroupedQueryAttention(nn.Module):
    def __init__(
        self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False, dtype=None
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        # 中文注释: Q 的输出维度是 num_heads*head_dim;K、V 的输出维度只有 num_kv_groups*head_dim,
        # 远小于 Q 的维度 —— 这正是 GQA 节省 KV 缓存显存的关键
        self.W_query = nn.Linear(d_in, self.d_out, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)

        self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)

        # 中文注释: QK-Norm —— 对每个头的 query/key 向量在 head_dim 维度上做 RMSNorm,
        # 用于稳定训练、抑制注意力 logits 数值爆炸,是 Qwen3 的关键设计之一
        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

    def forward(self, x, mask, cos, sin):
        b, num_tokens, _ = x.shape

        # 中文注释: 计算 Q、K、V 的线性投影;注意 K、V 的头数是 num_kv_groups,少于 Q 的 num_heads
        # Apply projections
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

        # 中文注释: 重排为多头形式并转置,形状变为 (b, num_heads 或 num_kv_groups, num_tokens, head_dim),
        # 把“头”维度换到 seq_len 之前,便于做批量矩阵乘法
        # Reshape
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # 中文注释: QK-Norm 在拆分出多头之后、应用 RoPE 之前进行,作用在每个头的 head_dim 维度上
        # Optional normalization
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys = self.k_norm(keys)

        # 中文注释: 对 Q、K 分别应用旋转位置编码;GQA 下 K 的头数虽然只有 num_kv_groups,
        # 但 cos/sin 只依赖 head_dim 与序列位置,与头数无关,可以直接复用
        # Apply RoPE
        queries = apply_rope(queries, cos, sin)
        keys = apply_rope(keys, cos, sin)

        # 中文注释: GQA 的核心操作 —— 把 K、V 沿“头”维度重复 group_size = num_heads/num_kv_groups 次,
        # 使其头数与 Q 对齐,从而可以直接做逐头注意力计算;
        # 而在存储/投影阶段 K、V 仍保持较少的头数,这正是节省显存的地方
        # Expand K and V to match number of heads
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        # 中文注释: 计算注意力得分 Q @ K^T,形状 (b, num_heads, num_tokens, num_tokens),
        # 用因果掩码屏蔽未来位置后,除以 sqrt(head_dim) 缩放,再做 softmax 得到注意力权重
        # Attention
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)

        # 中文注释: 注意力权重与 V 加权求和后,把多头维度换回并拼接,
        # 形状还原为 (b, num_tokens, num_heads*head_dim),再经 out_proj 投影回 d_in 维度
        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        return self.out_proj(context)
# 中文注释: Transformer 块 —— 采用 Pre-Norm 结构(先做 RMSNorm 再进注意力/前馈),
# 并用残差连接(shortcut)把归一化前的输入加回子层输出
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            head_dim=cfg["head_dim"],
            num_kv_groups=cfg["n_kv_groups"],
            qk_norm=cfg["qk_norm"],
            dtype=cfg["dtype"]
        )
        # 中文注释: num_experts > 0 时使用稀疏 MoE 前馈层,否则退化为普通稠密 SwiGLU 前馈层(FeedForward)
        if cfg["num_experts"] > 0:
            self.ff = MoEFeedForward(cfg)
        else:
            self.ff = FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=1e-6)

    def forward(self, x, mask, cos, sin):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x, mask, cos, sin)  # Shape [batch_size, num_tokens, emb_size]
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut  # Add the original input back

        return x
# 中文注释: 完整的 Qwen3 MoE 模型 —— 词嵌入 + N 层 TransformerBlock(内含 GQA 与 MoE)
# + 最终 RMSNorm + 输出线性头(out_head)
class Qwen3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Main model parameters
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        self.trf_blocks = nn.ModuleList(  # ModuleList since Sequential can only accept one input, and we need `x, mask, cos, sin`
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        # Reusable utilities
        if cfg["head_dim"] is None:
            head_dim = cfg["emb_dim"] // cfg["n_heads"]
        else:
            head_dim = cfg["head_dim"]
        # 中文注释: 整个模型只预计算一份 cos/sin 表并在所有层间共享,
        # 注册为非持久化(persistent=False)buffer,不会被保存进 state_dict
        cos, sin = compute_rope_params(
            head_dim=head_dim,
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.cfg = cfg


    def forward(self, in_idx):
        # Forward pass
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds

        num_tokens = x.shape[1]
        # 中文注释: 构造因果注意力掩码,上三角(不含对角线)为 True 表示需要被屏蔽(看不到未来)的位置,
        # 形状为 (num_tokens, num_tokens)
        mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1)

        for block in self.trf_blocks:
            x = block(x, mask, self.cos, self.sin)
        x = self.final_norm(x)
        # 中文注释: 输出头把隐藏状态映射到词表大小,得到每个位置在整个词表上的 logits
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits

2. Initialize model

In [ ]:
# Same config for
# 中文注释: 下面这份配置适用于多个 Qwen3-MoE 变体模型(仅模型名字不同,网络结构相同),见下方链接

# https://huggingface.co/Qwen/Qwen3-Coder-30B-A3B-Instruct (Qwen3 Coder Flash)
# https://huggingface.co/Qwen/Qwen3-30B-A3B-Thinking-2507
# https://huggingface.co/Qwen/Qwen3-235B-A22B-Instruct-2507
# https://huggingface.co/Qwen/Qwen3-30B-A3B (original Instruct/Thinking hybrid model)

# 中文注释: Qwen3-30B-A3B 系列 MoE 模型的结构超参数配置
QWEN3_CONFIG = {
    "vocab_size": 151_936,
    "context_length": 262_144,
    "emb_dim": 2048,
    # 中文注释: n_heads 是 query 头总数;下面的 n_kv_groups 是 key/value 头数,
    # 二者之比就是 GQA 的分组大小 group_size
    "n_heads": 32,
    "n_layers": 48,
    # 中文注释: head_dim 显式指定为 128,并不要求等于 emb_dim/n_heads(此处 2048/32=64),
    # 说明 Q/K/V 每个头的维度可以独立于 emb_dim 单独设置
    "head_dim": 128,
    # 中文注释: qk_norm 开启后,GroupedQueryAttention 会对每个头的 Q、K 做 RMSNorm(QK-Norm),
    # 这是 Qwen3 用来稳定训练的关键设计
    "qk_norm": True,
    # 中文注释: n_kv_groups=4 而 n_heads=32,即每 8 个 query 头共享 1 组 key/value 头
    # (GQA 分组大小 group_size = 32/4 = 8)
    "n_kv_groups": 4,
    # 中文注释: RoPE 的 base 值远大于常见的 10000,配合超长的 context_length(262144)使用,
    # 更大的 base 有助于在超长上下文下保持位置编码的区分度
    "rope_base": 10_000_000.0,
    "dtype": torch.bfloat16,
    # 中文注释: 以下为 MoE 相关参数 —— 模型共有 128 个专家
    "num_experts": 128,
    # 中文注释: 每个 token 只激活其中 8 个专家(top-8 路由),因此称为“稀疏”MoE
    "num_experts_per_tok": 8,
    # 中文注释: 每个专家内部 SwiGLU 的隐藏维度只有 768,远小于稠密 FeedForward 可能用到的 hidden_dim,
    # “大量专家 + 较小隐藏维度”是 MoE 扩大参数总量同时控制每个 token 计算量的方式
        "moe_hidden_dim": 768,
}
# 中文注释: 自动选择可用的计算设备 —— 优先 CUDA,其次 Apple Silicon 的 MPS,否则退回 CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(device)

In [ ]:
# 中文注释: 固定随机种子,保证模型参数初始化过程可复现
torch.manual_seed(123)

# 中文注释: 利用 torch.device 上下文管理器,让模型内部新建的所有张量直接分配到目标设备上,
# 避免“先在默认设备创建整份模型、再整体搬运”带来的显存峰值和额外开销
with device:
    model = Qwen3Model(QWEN3_CONFIG)

# 中文注释: 因为上面已经在 with device 语境下创建了模型,这里无需再执行 model.to(device)
#model.to(device)

In [ ]:
# 中文注释: 用形状为 (1, 3) 的示例输入做一次前向传播,快速检查模型各层的形状是否搭建正确
model(torch.tensor([1, 2, 3]).unsqueeze(0).to(device))

In [ ]:
# 中文注释: 统计模型全部参数的元素总数
# (若输出头与词嵌入权重共享,即 weight tying,这里会把同一份参数重复计入一次)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# Account for weight tying
# 中文注释: 因此这里减去重复计入的 tok_emb 参数量,得到真正“独立”的参数总量
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

In [ ]:
# 中文注释: 粗略估算模型在给定 dtype 下的显存占用 —— 假设参数、梯度都按同一种 dtype 存储,
# 显存 ≈ (参数量 + 梯度量 + buffer 元素数量) × 每个元素的字节数
def calc_model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

# 中文注释: 对比 float32 与 bfloat16 两种精度下的模型体积(bfloat16 约为 float32 的一半)
print(f"float32 (PyTorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

4. Load pretrained weights

In [ ]:
# 中文注释: 把从 HuggingFace 下载的原始权重字典(safetensors,键名遵循 HF Qwen3 命名规范)
# 逐一拷贝进当前手搭模型(Qwen3Model)对应的参数张量里
def load_weights_into_qwen(model, param_config, params):
    # 中文注释: 内部工具函数 —— 先校验形状一致,再把右侧权重原地拷贝进左侧参数
    # (不改变参数对象本身,只更新其数值,因此可以直接赋值回 nn.Parameter)
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}")

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    # 中文注释: 加载词嵌入权重
    model.tok_emb.weight = assign(model.tok_emb.weight, params["model.embed_tokens.weight"], "model.embed_tokens.weight")

    for l in range(param_config["n_layers"]):
        block = model.trf_blocks[l]
        att = block.att

        # Q, K, V projections
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight"
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight"
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight"
        )

        # Output projection
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight"
        )

        # 中文注释: 仅当模型启用了 qk_norm 时才存在 q_norm/k_norm 子模块,需要单独加载它们的 scale 参数
        # QK norms
        if hasattr(att, "q_norm") and att.q_norm is not None:
            att.q_norm.scale = assign(
                att.q_norm.scale,
                params[f"model.layers.{l}.self_attn.q_norm.weight"],
                f"model.layers.{l}.self_attn.q_norm.weight"
            )
        if hasattr(att, "k_norm") and att.k_norm is not None:
            att.k_norm.scale = assign(
                att.k_norm.scale,
                params[f"model.layers.{l}.self_attn.k_norm.weight"],
                f"model.layers.{l}.self_attn.k_norm.weight"
            )

        # Attention layernorm
        block.norm1.scale = assign(
            block.norm1.scale,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight"
        )

        # 中文注释: MoE 权重命名与稠密模型不同 —— 路由门控是 mlp.gate.weight,
        # 每个专家的三个投影分别是 experts.{e}.gate_proj/up_proj/down_proj,需要按专家下标循环加载
        # Feedforward weights
        if "num_experts" in param_config and param_config["num_experts"] > 0:
            # Load router (gating) weights
            block.ff.gate.weight = assign(
                block.ff.gate.weight,
                params[f"model.layers.{l}.mlp.gate.weight"],
                f"model.layers.{l}.mlp.gate.weight"
            )
            # Load expert weights
            # 中文注释: 按专家编号依次加载该专家的三个线性层权重
            # (分别对应本 notebook 中 MoEFeedForward 的 fc1/fc2/fc3)
            for e in range(param_config["num_experts"]):
                prefix = f"model.layers.{l}.mlp.experts.{e}"
                block.ff.fc1[e].weight = assign(
                    block.ff.fc1[e].weight,
                    params[f"{prefix}.gate_proj.weight"],
                    f"{prefix}.gate_proj.weight"
                )
                block.ff.fc2[e].weight = assign(
                    block.ff.fc2[e].weight,
                    params[f"{prefix}.up_proj.weight"],
                    f"{prefix}.up_proj.weight"
                )
                block.ff.fc3[e].weight = assign(
                    block.ff.fc3[e].weight,
                    params[f"{prefix}.down_proj.weight"],
                    f"{prefix}.down_proj.weight"
                )

        else:
            # 中文注释: 非 MoE(稠密)分支 —— 直接加载单一的 FeedForward 三个投影权重
            block.ff.fc1.weight = assign(
                block.ff.fc1.weight,
                params[f"model.layers.{l}.mlp.gate_proj.weight"],
                f"model.layers.{l}.mlp.gate_proj.weight"
            )
            block.ff.fc2.weight = assign(
                block.ff.fc2.weight,
                params[f"model.layers.{l}.mlp.up_proj.weight"],
                f"model.layers.{l}.mlp.up_proj.weight"
            )
            block.ff.fc3.weight = assign(
                block.ff.fc3.weight,
                params[f"model.layers.{l}.mlp.down_proj.weight"],
                f"model.layers.{l}.mlp.down_proj.weight"
            )

        block.norm2.scale = assign(
            block.norm2.scale,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight"
        )

    # Final normalization and output head
    model.final_norm.scale = assign(model.final_norm.scale, params["model.norm.weight"], "model.norm.weight")

    # 中文注释: 部分 Qwen3 变体的输出头与词嵌入权重共享(weight tying),
    # 此时权重文件中不含独立的 lm_head.weight,需要手动把 out_head 指向 tok_emb 的权重
    if "lm_head.weight" in params:
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    else:
        model.out_head.weight = model.tok_emb.weight
        print("Model uses weight tying.")
# 中文注释: 以下开始从 HuggingFace Hub 下载指定的 Qwen3-MoE 预训练权重(safetensors 分片),
# 并调用上面定义的 load_weights_into_qwen 把权重灌入前面搭建好的模型
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import snapshot_download

# 中文注释: 以下四行重复给 repo_id 赋值,实际生效的是最后一行(Qwen3-Coder-30B-A3B-Instruct);
# 如需切换到其他模型变体,注释掉不需要的行、保留想用的那一行即可
repo_id = "Qwen/Qwen3-30B-A3B"  # Original Instruct/Thinking hybrind model
repo_id = "Qwen/Qwen3-235B-A22B-Instruct-2507"  # New instruct model
repo_id = "Qwen/Qwen3-30B-A3B-Thinking-2507"  # New thinking model
repo_id = "Qwen/Qwen3-Coder-30B-A3B-Instruct"  # (Qwen3 Coder Flash)

local_dir = Path(repo_id).parts[-1]

# 中文注释: 下载模型仓库到本地目录,并读取 safetensors 的分片索引文件
repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
index_path = os.path.join(repo_dir, "model.safetensors.index.json")
with open(index_path, "r") as f:
    index = json.load(f)

# 中文注释: 遍历所有分片文件,把权重逐个加载并合并进一个统一的字典
weights_dict = {}
for filename in set(index["weight_map"].values()):
    shard_path = os.path.join(repo_dir, filename)
    shard = load_file(shard_path)
    weights_dict.update(shard)

# 中文注释: 调用前面定义的函数把下载好的权重灌入模型,并将模型整体搬到目标计算设备
load_weights_into_qwen(model, QWEN3_CONFIG, weights_dict)
model.to(device);

3. Load tokenizer

In [ ]:
# 中文注释: 基于 HuggingFace tokenizers 库的 BPE 分词器,
# 包装出与 Qwen3 聊天模板(chat template)配套的编码/解码接口
import re
from tokenizers import Tokenizer

class Qwen3Tokenizer:
    # 中文注释: Qwen3 词表中的特殊 token 列表,包括对话标记 <|im_start|>/<|im_end|>、
    # 多模态占位符,以及推理标记 <think>/</think> 等
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
        "<think>", "</think>"
    ]
    # 中文注释: 用于在编码前把文本中出现的特殊 token 从普通文本中切分出来,
    # 避免它们被底层 BPE 当成普通字符串继续做子词切分
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>|<think>|</think>)")

    def __init__(self, tokenizer_file_path="tokenizer.json", repo_id=None,
                 apply_chat_template=True, add_generation_prompt=False, add_thinking=False):

        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        self._special_to_id = {}
        for t in self._SPECIALS:
            tid = self._tok.token_to_id(t)
            if tid is not None:
                self._special_to_id[t] = tid

        # 中文注释: <|endoftext|> 同时用作 pad token,并作为默认的 eos token
        self.pad_token_id = self._special_to_id["<|endoftext|>"]
        self.eos_token_id = self.pad_token_id

        # 中文注释: 非 Base(即经过指令/对话微调)的模型使用 <|im_end|> 作为对话轮次结束符
        if repo_id and "Base" not in repo_id:
            eos_token = "<|im_end|>"
        else:
            eos_token = "<|endoftext|>"
        if eos_token in self._special_to_id:
            self.eos_token_id = self._special_to_id[eos_token]

    def encode(self, text, chat_wrapped=None):
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        # 中文注释: 如果输入整体正好是某个特殊 token(且不含换行),直接返回其 id,不走常规分词流程
        stripped = text.strip()
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        # 中文注释: 按 Qwen3 的对话模板包裹用户输入(加上 <|im_start|>user ... <|im_end|> 等标记)
        if chat_wrapped:
            text = self._wrap_chat(text)

        # 中文注释: 先用正则把特殊 token 从文本中切分出来;普通文本片段交给底层 BPE 分词器编码,
        # 特殊片段则直接映射为预先记录好的 token id
        ids = []
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)

    def _wrap_chat(self, user_msg):
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant"
            if self.add_thinking:
                s += "\n"
            else:
                s += "\n<think>\n\n</think>\n\n"
        return s
# 中文注释: 实例化分词器,并对示例 prompt 做编码/解码,验证聊天模板与分词器工作是否正常
tokenizer_file_path = f"{Path(repo_id).parts[-1]}/tokenizer.json"

tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=tokenizer_file_path,
    repo_id=repo_id,
    apply_chat_template=True,
    add_generation_prompt=True,
    add_thinking=True
)
# prompt = "Give me a short introduction to large language models."
prompt = "Implement a binary search function in Python"


# 中文注释: 对提示词编码为 token id 序列,再解码回文本,用于人工检查聊天模板是否被正确应用
input_token_ids = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids)
text

4. Generate text

In [ ]:
# 中文注释: 基础的自回归文本生成 —— 逐 token 贪心解码,并以流式方式 yield 每个新生成的 token。
# 注意: 该实现未使用 KV 缓存,每一步都会把完整的历史 token 序列重新喂给模型做前向传播,
# 因此生成速度较慢,序列越长单步耗时越大
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):

    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # 中文注释: 对当前完整序列做一次前向传播,只取最后一个位置的 logits 用于预测下一个 token,
            # 形状为 (batch, vocab_size)
            out = model(token_ids)[:, -1]
            next_token = torch.argmax(out, dim=-1, keepdim=True)
            # 中文注释: 贪心解码 —— 直接取概率(logits)最大的 token,不做随机采样

            if (eos_token_id is not None
                   and torch.all(next_token == eos_token_id)):
               break

            yield next_token

            # 中文注释: 把新生成的 token 拼接到序列末尾,作为下一步的输入
            # (因为没有 KV 缓存,每步输入长度都会增加 1,需要整段重新计算)
            token_ids = torch.cat([token_ids, next_token], dim=1)
# 中文注释: 准备好初始输入 token 序列(形状 (1, prompt_len)),并放到目标计算设备上
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)


# 中文注释: 逐 token 生成并流式打印结果 —— 每生成一个 token 就立即解码并输出,不必等全部生成完毕
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=100,  # Cut-off after 100 tokens because non-kv variant is very slow
    # eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )